L'idée : C'est que le client fournit un audio( vocale) le chiffre en cryptage fully homomrophique , et l'envoie au serveur  :

Le serveur ferra deux manipulations intéressantes:
- calcule la durée du silence de l'audio  et la durée du son  
-comparer le contenue de deux audios sans avoir accès  à ces audios !


In [4]:
import scipy as sp
import numpy as np
import librosa
import tenseal as ts
import utils
import soundfile as sf


In [ ]:
# audio_artifiel_pomme_java, sr = librosa.load('my_data/artificiel_pomme_java.mp3', sr=None)
audio_2s_delay,sd = librosa.load('my_data/python_banane_ia_2sdelay.mp3', sr=None)
audio_6s_delay,sde = librosa.load('my_data/cetic_test_6delay.mp3', sr=None)
audio_10s_delay,sd3 = librosa.load('my_data/10_s_pure_silence.mp3', sr=None)
audio_0_delay,sd2 = librosa.load('my_data/no_silence.mp3', sr=None)

print("audio",audio_2s_delay)
# 184896
print("audio shape",len(audio_2s_delay))
# print("sr",sr)
print(f"Taux d'échantillonnage: {sde}")
# print(f"Nombre d'échantillons: {len(audio_artifiel_pomme_java)}")

context =  ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
context.generate_galois_keys()
context.global_scale = 2 ** 40
my_secret_key = context.serialize(save_secret_key=True)
utils.write_data("Keys_provider/secret_key.txt", my_secret_key)
context.make_context_public()
public_key=context.serialize()
utils.write_data("Keys_provider/public_key.txt", public_key)

for i in range(len(audio_0_delay)):
    audio_0_delay[i] = audio_0_delay[i] * 1000

for i in range(len(audio_10s_delay)):
    audio_10s_delay[i] = audio_10s_delay[i] * 1000


for i in range(len(audio_6s_delay)):
    audio_6s_delay[i] = audio_6s_delay[i] * 1000

for  i in range(len(audio_2s_delay)):
    audio_2s_delay[i] = audio_2s_delay[i] *1000
print("audio_2s_delay",audio_2s_delay)    

audio_6s_delay_encrypted = ts.ckks_vector(context, audio_6s_delay)
audio_2s_delay_encrypted = ts.ckks_vector(context, audio_2s_delay)
audio_0_delay_encrypted = ts.ckks_vector(context, audio_0_delay)
audio_10s_delay_encrypted = ts.ckks_vector(context, audio_10s_delay)

# audio_artifiel_pomme_java_encrypted = ts.ckks_vector(context, audio_artifiel_pomme_java)



utils.write_data("output_provider/python_banane_ia_2sdelay_encrypted.txt", audio_2s_delay_encrypted.serialize())
utils.write_data("output_provider/cetic_test_6sdelay_encrypted.txt", audio_6s_delay_encrypted.serialize())
# utils.write_data("output_provider/artificiel_pomme_java_encrypted.txt", audio_artifiel_pomme_java_encrypted.serialize())
utils.write_data("output_provider/no_silence_encrypted.txt", audio_0_delay_encrypted.serialize())
utils.write_data("output_provider/10_s_pure_silence_encrypted.txt", audio_10s_delay_encrypted.serialize())


audio [ 0.0000000e+00  4.8991182e-13  2.7853110e-13 ... -3.2327348e-06
 -1.8873980e-05 -1.2278676e-05]
audio shape 184896
Taux d'échantillonnage: 24000
audio_2s_delay [ 0.0000000e+00  4.8991183e-10  2.7853111e-10 ... -3.2327347e-03
 -1.8873980e-02 -1.2278676e-02]
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you 

In [11]:
loaded_secret_key = utils.read_data("Keys_provider/secret_key.txt")
secret_key = ts.context_from(loaded_secret_key).secret_key()
seuil = 0.9999  
sample_rate = 24000
#--------------------------1
totale_de_silence = utils.read_data("uploads/silence_result_encrypted1.txt")
totale_de_silence = ts.lazy_ckks_vector_from(totale_de_silence)
totale_de_silence.link_context(context)
sigmoid_decrypted = totale_de_silence.decrypt(secret_key)


nb_silencieux = sum(1 for v in sigmoid_decrypted if v > seuil)

duree_silence = nb_silencieux / sample_rate
#--------------------------2
totale_de_silence2 = utils.read_data("uploads/silence_result_encrypted2.txt")
totale_de_silence2 = ts.lazy_ckks_vector_from(totale_de_silence2)
totale_de_silence2.link_context(context)
sigmoid_decrypted2 = totale_de_silence2.decrypt(secret_key)


nb_silencieux2 = sum(1 for v in sigmoid_decrypted2 if v > seuil)

duree_silence2 = nb_silencieux2 / sample_rate
#--------------------------3
totale_de_silence3 = utils.read_data("uploads/10_s_pure_silence_encrypted3.txt")
totale_de_silence3 = ts.lazy_ckks_vector_from(totale_de_silence3)
totale_de_silence3.link_context(context)
sigmoid_decrypted3 = totale_de_silence3.decrypt(secret_key)


nb_silencieux3 = sum(1 for v in sigmoid_decrypted3 if v > seuil)

duree_silence3 = nb_silencieux3 / sample_rate


#--------------------------4
totale_de_silence2 = utils.read_data("uploads/no_silence_encrypted4.txt.txt")
totale_de_silence2 = ts.lazy_ckks_vector_from(totale_de_silence2)
totale_de_silence2.link_context(context)
sigmoid_decrypted2 = totale_de_silence2.decrypt(secret_key)


nb_silencieux4 = sum(1 for v in sigmoid_decrypted2 if v > seuil)
duree_silence4 = nb_silencieux4 / sample_rate


#-----------------------result

print("--------------------------------------------")
print("Durée du silence de l'audio 1 sans passer par du cryptage homorphique : 4.97 secondes")
print(f"Durée estimée du silence audio 1 en passant par le cryptage homorphique :{duree_silence:.3f} secondes")
print("\n--------------------------------------------\n")
print("Durée du silence de l'audio 2 sans passer par du cryptage homorphique : 6.68 secondes")
print(f"Durée estimée du silence audio 2 en passant par le cryptage homorphique : {duree_silence2:.3f} secondes")
print("\n--------------------------------------------\n")
print("Durée du silence de l'audio 3 sans passer par du cryptage homorphique : 10.06 secondes")
print(f"Durée estimée du silence audio 3 en passant par le cryptage homorphique : {duree_silence3:.3f} secondes")
print("\n--------------------------------------------\n")
print("Durée du silence de l'audio 4 sans passer par du cryptage homorphique : 0.00 secondes")
print(f"Durée estimée du silence audio 4 en passant par le cryptage homorphique : {duree_silence4:.3f} secondes")


--------------------------------------------
Durée du silence de l'audio 1 sans passer par du cryptage homorphique : 4.97 secondes
Durée estimée du silence audio 1 en passant par le cryptage homorphique :5.030 secondes

--------------------------------------------

Durée du silence de l'audio 2 sans passer par du cryptage homorphique : 6.68 secondes
Durée estimée du silence audio 2 en passant par le cryptage homorphique : 6.653 secondes

--------------------------------------------

Durée du silence de l'audio 3 sans passer par du cryptage homorphique : 10.06 secondes
Durée estimée du silence audio 3 en passant par le cryptage homorphique : 10.056 secondes

--------------------------------------------

Durée du silence de l'audio 4 sans passer par du cryptage homorphique : 0.00 secondes
Durée estimée du silence audio 4 en passant par le cryptage homorphique : 0.078 secondes
